In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q1_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> grayscale((0, 0, 0))\n(0, 0, 0)',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> grayscale((10, 20, 30))\n(20, 20, 20)',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> grayscale((255, 255, 255))\n(255, 255, 255)',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> grayscale((1, 1, 0))\n(0, 0, 0)',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 3,
        'suites': [{'cases': [{'code': '>>> apply_filter([[(10, 20, 30)]])\n[[(20, 20, 20)]]',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> apply_filter([[(0, 0, 0), (255, 255, 255)], [(3, '
                                       '3, 3), (9, 0, 0)]])\n'
                                       '[[(0, 0, 0), (255, 255, 255)], [(3, 3, 3), (3, 3, 3)]]',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> apply_filter([])\n[]',
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> brighten([[(10, 20, 30)]], 5)\n[[(15, 25, 35)]]',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> brighten([[(250, 0, 0)]], 10)\n[[(255, 10, 10)]]',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> brighten([[(1, 2, 3), (4, 5, 6)]], 0)\n'
                                       '[[(1, 2, 3), (4, 5, 6)]]',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> brighten([], 5)\n[]',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q5': {'name': 'q5',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> top_posts([[12, 'mika'], [40, 'ade'], [12, "
                                       "'zo']])\n"
                                       "[[40, 'ade'], [12, 'zo'], [12, 'mika']]",
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> def t5():\n'
                                       "...     data = [[12, 'mika'], [40, 'ade']]\n"
                                       '...     out = top_posts(data)\n'
                                       "...     return out == [[40, 'ade'], [12, 'mika']] and "
                                       "data == [[12, 'mika'], [40, 'ade']]\n"
                                       '>>> t5()\n'
                                       'True',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> top_posts([])\n[]',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q6': {'name': 'q6',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> q6_seconds == [2820, 3660, 3300]\nTrue',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('12-Tuples-Comprehensions', _TESTS)
check = grader.check

print("Ready. 12-Tuples-Comprehensions loaded: 6 question(s), 12 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

# Homework: Tuples and Comprehensions — Photo Filters

Every filter you have ever tapped is arithmetic on a grid of numbers.

A digital image is a **grid of pixels**. Each pixel is three numbers between 0 and 255 —
how much red, how much green, how much blue. `(255, 0, 0)` is pure red, `(0, 0, 0)` is
black, `(255, 255, 255)` is white.

In Python we store one pixel as a **tuple**, `(r, g, b)`, and a whole image as a **list of
lists of tuples** — the outer list holds rows, and each row holds the pixels across it:

```python
image = [[(255, 0, 0), (0, 255, 0)],       # row 0: red,  green
         [(0, 0, 255), (0, 0, 0)]]         # row 1: blue, black
```

`image[1][0]` is the pixel in row 1, column 0 — the blue one.

In this homework you will write two filters and a leaderboard, and you will meet the two
new tools this topic brings: **tuples**, and **comprehensions**.

## Question 1 — Why a tuple and not a list?

```python
pixel = (255, 0, 0)
pixel[0] = 0
```

What happens on that second line?

**A** — `pixel` becomes `(0, 0, 0)`

**B** — Python raises a `TypeError`, because a tuple cannot be changed once it is created

**C** — `pixel` becomes the list `[0, 0, 0]`

Set `q1_option` to `'A'`, `'B'` or `'C'`.

That immutability is the whole reason a pixel is a tuple and not a list. A pixel is one
value made of three parts, like a coordinate or a date. Three numbers that belong
together and should never be edited one at a time — if you want a different colour, you
make a different pixel. A list is for a collection that grows, shrinks and gets
rearranged, which is what a *row* of pixels is.

In [ ]:
q1_option = ...

In [ ]:
grader.check("q1")

## Question 2 — Greyscale one pixel

The simplest filter there is: average the three channels, and give all three the same
value.

Write `grayscale(pixel)`, which takes a `(r, g, b)` tuple and returns a new tuple where
every channel is the **integer** average `(r + g + b) // 3`.

```
grayscale((0, 0, 0))        ->  (0, 0, 0)
grayscale((10, 20, 30))     ->  (20, 20, 20)
grayscale((255, 255, 255))  ->  (255, 255, 255)
grayscale((1, 1, 0))        ->  (0, 0, 0)        2 // 3 is 0
```

Use `//`, not `/`. Channels are whole numbers — there is no such thing as 19.5 red.

You can pull the three parts out with indexing, `pixel[0]`, or by **unpacking**:

```python
r, g, b = pixel
```

Both are fine. The second is easier to read, and reading is most of what code is for.

In [ ]:
def grayscale(pixel):
    ...

# try it out
print(grayscale((10, 20, 30)))

In [ ]:
grader.check("q2")

## Question 3 — Greyscale a whole image

Now run that filter over every pixel in an image. Write `apply_filter(image)`, which takes
a 2D list of pixels and returns a **new** 2D list with `grayscale` applied to each one.
Leave the original image alone.

```
apply_filter([[(10, 20, 30)]])
    ->  [[(20, 20, 20)]]

apply_filter([[(0, 0, 0), (255, 255, 255)], [(3, 3, 3), (9, 0, 0)]])
    ->  [[(0, 0, 0), (255, 255, 255)], [(3, 3, 3), (3, 3, 3)]]

apply_filter([])
    ->  []
```

This is the standard shape for walking a grid — **a loop inside a loop**:

- the **outer** loop takes one row at a time
- the **inner** loop takes one pixel at a time out of that row

Build a fresh list for each row, append the filtered pixels into it, then append the
finished row to the output image.

Notice you already wrote the hard part. `apply_filter` calls `grayscale`; it does not
redo the arithmetic. Each case is worth 1 point.

In [ ]:
def apply_filter(image):
    ...

# try it out
photo = [[(0, 0, 0), (255, 255, 255)],
         [(3, 3, 3), (9, 0, 0)]]
print(apply_filter(photo))

In [ ]:
grader.check("q3")

## Question 4 — Brighten, with a comprehension

Write `brighten(image, amount)`, which adds `amount` to every channel of every pixel —
but never goes above 255, because 255 is as bright as a channel gets.

```
brighten([[(10, 20, 30)]], 5)             ->  [[(15, 25, 35)]]
brighten([[(250, 0, 0)]], 10)             ->  [[(255, 10, 10)]]     red stops at 255
brighten([[(1, 2, 3), (4, 5, 6)]], 0)     ->  [[(1, 2, 3), (4, 5, 6)]]
brighten([], 5)                           ->  []
```

**Write this one as a list comprehension**, not as nested `for` loops with `.append()`.

A comprehension is the same loop written as one expression:

```python
doubled = [n * 2 for n in numbers]                    # every number, doubled
big = [n * 2 for n in numbers if n > 10]              # only the ones over 10
```

An image needs two of them, one inside the other — the outer over rows, the inner over
the pixels in a row:

```python
[[ something(pixel) for pixel in row ] for row in image]
```

`min()` is how you stop at 255: `min(255, r + amount)` gives 255 whenever the sum would go
past it, and the sum otherwise.

In [ ]:
def brighten(image, amount):
    ...

# try it out
print(brighten([[(250, 0, 0)]], 10))

In [ ]:
grader.check("q4")

## Question 5 — The leaderboard

Your app shows the most-liked posts of the day. Each post is a small list,
`[likes, handle]`, and you have a list of them:

```python
posts = [[12, 'mika'], [40, 'ade'], [12, 'zo']]
```

Write `top_posts(posts)`, which returns a **new** list of the same posts sorted **most
liked first**, without changing the list it was given.

```
top_posts([[12, 'mika'], [40, 'ade'], [12, 'zo']])
    ->  [[40, 'ade'], [12, 'zo'], [12, 'mika']]

top_posts([])  ->  []
```

Look closely at what happened to the two 12s. Python compares lists **lexicographically** —
the way a dictionary orders words. It compares the first items; only if those are equal
does it look at the second. So `likes` decides the order, and `handle` silently breaks the
ties. `'zo'` landing ahead of `'mika'` is that tie-break running in reverse, and it is why
putting `likes` first in the list is a design decision, not an accident.

`sorted(posts, reverse=True)` does the whole thing. Use it rather than `posts.sort()`, so
the caller's list survives.

In [ ]:
def top_posts(posts):
    ...

# try it out
today = [[12, 'mika'], [40, 'ade'], [12, 'zo']]
print(top_posts(today))
print(today)    # should be unchanged

In [ ]:
grader.check("q5")

## Question 6 — Turn a loop into a comprehension

Here is a loop that finds every video longer than half an hour and converts its length to
seconds:

```python
minutes = [3, 47, 12, 61, 8, 55]

long_ones = []
for m in minutes:
    if m > 30:
        long_ones.append(m * 60)
```

Write the same thing as a **single list comprehension** and assign it to `q6_seconds`.

Every comprehension has the same three parts, and they are the same three parts as the
loop above, reordered:

```
[  m * 60     for m in minutes     if m > 30  ]
   ^^^^^^     ^^^^^^^^^^^^^^^^     ^^^^^^^^^^
   what you   what you loop        which ones
   collect    over                 you keep
```

Write it out, then run the loop version alongside it and confirm the two lists match.

In [ ]:
minutes = [3, 47, 12, 61, 8, 55]

q6_seconds = ...

print(q6_seconds)

In [ ]:
grader.check("q6")

## Beyond the homework

Not graded:

- Write `flip(image)`, mirroring the image left-to-right. A slice with a step of `-1` on
  each row does it in one comprehension.
- Write `invert(pixel)`, returning `(255 - r, 255 - g, 255 - b)`, then run it over an image
  with `apply_filter`'s shape. What would you have to change so that `apply_filter` could
  take *any* pixel filter as an argument?
- `min(255, r + amount)` clamps the top. What stops `brighten(image, -300)` going below
  zero? Fix it, and write the test case that proves it.

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

- Ensure you run all the cells in order from the top to the bottom
- Run the final cell to see your total and produce your receipt
- **File->Download->Download .ipynb** and upload the `.ipynb` file to Moodle

Your mark comes from your instructor re-running this notebook, not from the receipt
line -- so make sure the notebook itself is saved with your work in it.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
